In [ ]:
# 1. Cargar las tablas desde PostgreSQL
df_v = read_pg("public.venta")
df_dv = read_pg("public.detalle_venta")
df_cv = read_pg("public.canal_venta")

# 2. Realizar los JOINs en PySpark con las columnas reales
df_ventas_raw = (
    df_v.alias("v")
    .join(df_dv.alias("dv"), on="id_venta", how="inner")
    .join(df_cv.alias("cv"), on="id_canal_venta", how="inner")
    .select(
        "v.id_venta",
        "v.id_cliente",
        "v.id_supermercado",
        "v.id_canal_venta",
        "v.id_empleado",
        "v.fecha_hora_venta",
        "v.total_venta",
        "v.descuento_venta",
        "v.impuesto_venta",
        "dv.id_detalle_venta",
        "dv.id_producto",
        "dv.cantidad",
        "dv.precio_unitario",
        "dv.descuento_unitario",
        "dv.subtotal_linea",
        "cv.nombre_canal",
        "cv.descripcion_canal",
        "cv.estado_canal",
    )
)

# 3. Guardar en Delta Lake (Capa Bronze)
df_ventas_raw.write.mode("overwrite").saveAsTable(T_VENTAS_RAW)

print("Bronze ventas_raw:", spark.table(T_VENTAS_RAW).count(), "filas")
display(spark.table(T_VENTAS_RAW).limit(10))